In [2]:
!which python3
!python3 --version

/opt/anaconda3/bin/python3
Python 3.12.12


Staff Schedule Upload

Description

Parse night staff schedule CSV files with expected format and generate output SQL insert statements
Only generates SQL text and does not execute the inserts

This is based on a Python port of updated PHP code
(which is a standalone CLI port of the original PHP uploader logic)

TODOs (carried over from PHP version):
- Ignore obvious blank content lines and header.
- Check for sequential dates with no gaps.
- Warn unknown initials
- Get alias mapping from db?
- Add a confirm after parse to do the actual inserts and/or write to sql file
- Make enough validation to allow someone like Gloria to use
- Create cron job that sends out reminder if db is about out of dates.
- Could be easy to select the wrong radio type, so grouping initials mapping by type and only looking at that grouping.

Other:
- open file picker to select input csv file (or xlsx file, tbd)
- auto-detect schedule type based on file name, user input, or contents
- auto-detect default date range based on file name, user input, or contents

In [ ]:
#!/usr/bin/env /opt/anaconda3/bin/python3

from __future__ import annotations

# import argparse  # tbd
import csv
import sys
from datetime import datetime
from pathlib import Path
from typing import Iterable

# notify root level of project so that common modules can be imported
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "common").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common import parse_split_cell

In [43]:
INITIALS: dict[str, str] = {
    "CJ": "cjordan",
    "TS": "tstickel",
    "JA": "jaycock",
    "CW": "cwilburn",
    "JRK": "julierk",
    "HH": "hhershley",
    "JP": "jpelletier",
    "TR": "tridenour",
    "AH": "ahatakeyama",
    "AR": "arostopchina",
    "RM": "rmorris",
    "LF": "lfuhrman",
    "MW": "mwahl",
    "MP": "mpiper",
    "SJ": "sjoseph",
    "NJ": "njordan",
    "JLP": "jlapinta",
    "TC": "tconnors",
    "DO": "dorr",
    "KWB": "kbrennon",
    "AD": "adeverse",
    "LS": "lsato",
    "EL": "elevine",
    "SG": "sguerpo",
    "MM": "mmedeiros",
    "CB": "cbishop",
    "KK": "kkameoka-vincent",
}

INITIALS_SA: dict[str, str] = {
    "PG": "pgomez",
    "CA": "calvarez",
    "RC": "randyc",
    "GD": "gdoppmann",
    "JL": "jlyke",
    "JW": "jwalawender",
    "SY": "syeh",
    "ML": "mlundquist",
    "RM": "rmcgurk",
    "LA": "lalcorn",
    "KM": "kmatthews",
    "PK": "pkrishnamoorthy",
}

In [9]:
# prints night staff and SA staff initials for verification
print(f'Night Staff: {INITIALS.keys()}')
print(f'SA Staff: {INITIALS_SA.keys()}')

Night Staff: dict_keys(['CJ', 'TS', 'JA', 'CW', 'JRK', 'HH', 'JP', 'TR', 'AH', 'AR', 'RM', 'LF', 'MW', 'MP', 'SJ', 'NJ', 'JLP', 'TC', 'DO', 'KWB', 'AD', 'LS', 'EL', 'SG', 'MM', 'CB', 'KK'])
SA Staff: dict_keys(['PG', 'CA', 'RC', 'GD', 'JL', 'JW', 'SY', 'ML', 'RM', 'LA', 'KM', 'PK'])


In [48]:
# command line argument parsing - skip

# import argparse
# from pathlib import Path

# def parse_args() -> argparse.Namespace:
#     parser = argparse.ArgumentParser(
#         description="Convert a staff schedule CSV into SQL insert statements."
#     )
#     parser.add_argument(
#         "--type",
#         required=True,
#         choices=["oa", "na", "sa", "eeoc"],
#         help="Schedule format to parse.",
#     )
#     parser.add_argument(
#         "--input",
#         required=True,
#         type=Path,
#         help="Input CSV path.",
#     )
#     parser.add_argument(
#         "--output",
#         type=Path,
#         help="Optional output SQL file path. Default: stdout.",
#     )
#     parser.add_argument(
#         "--verbose",
#         action="store_true",
#         help="Parse progress to stderr.",
#     )
#     return parser.parse_args()


In [51]:
# utility functions

def read_lines(path: Path) -> list[str]:
    return path.read_text(encoding="utf-8", errors="replace").splitlines()


def parse_csv_line(line: str) -> list[str]:
    row = next(csv.reader([line]))
    return [item.strip() for item in row]


def parse_date(value: str) -> str:
    text = value.strip()
    formats = (
        "%Y-%m-%d",
        "%m/%d/%Y",
        "%m/%d/%y",
        "%b %d %Y",
        "%b %d, %Y",
        "%B %d %Y",
        "%B %d, %Y",
    )
    for fmt in formats:
        try:
            return datetime.strptime(text, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    # Fall back to datetime parsing for ISO-like variants.
    try:
        return datetime.fromisoformat(text).strftime("%Y-%m-%d")
    except ValueError as exc:
        raise ValueError(f"Unable to parse date: {value!r}") from exc


def parse_date_to_obj(value: str):
    text = value.strip()
    try:
        return datetime.strptime(text, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(
            f"Invalid date format for date range: {value!r}. Expected YYYY-MM-DD."
        ) from exc


def in_date_range(date_value: str, start_date=None, end_date=None) -> bool:
    date_obj = parse_date_to_obj(date_value)
    if start_date is not None and date_obj < start_date:
        return False
    if end_date is not None and date_obj > end_date:
        return False
    return True


def sql_insert(date: str, telnr: str | int, alias: str, typ: str) -> str:
    # outgoing SQL always uses YYYY-MM-DD for MySQL date values.
    normalized_date = parse_date(str(date))
    return (
        "insert into nightStaff set "
        f"Date='{normalized_date}', "
        f"TelNr='{telnr}', "
        f"Alias='{alias}', "
        f"Type='{typ}';"
    )


def generate_output(lines: Iterable[str], output: Path | None) -> None:
    text = "\n".join(lines)
    if output is None:
        print(text)
        return
    output.write_text(text + "\n", encoding="utf-8")



In [53]:
# conversion functions for each schedule type

def convert_oa(lines: list[str], verbose: bool, start_date=None, end_date=None) -> list[str]:
    skip = {
        "",
        "X",
        "x",
        "L",
        "T",
        "H",
        "OM",
        "HQ",
        "PD",
        "SD",
        "CDP",
        "CPR",
        "ELP",
        "KSM",
        "PR",
        "First Aid",
        "SMOWG",
        "TelSched",
    }
    for i in range(1, 100):
        skip.add(f"O{i}")
        skip.add(f"o{i}")

    code = {
        "K1": "oa",
        "K2": "oa",
        "R1": "oar",
        "R2": "oar",
        "K1O": "oao",
        "K2O": "oao",
        "R1O": "oaro",
        "R2O": "oaro",
        "K1T": "oat",
        "K2T": "oat",
        "R1T": "oart",
        "R2T": "oart",
    }

    header: list[str] = []
    out: list[str] = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if verbose:
            print(f"OA line: {line}", file=sys.stderr, flush=True)
        line_parts = parse_csv_line(line)

        if "Date,DOW" in line:
            header = [h for h in line_parts if h]
            continue

        date = ""
        for key, hdr in enumerate(header):
            if key >= len(line_parts):
                continue
            cell = line_parts[key]
            if hdr == "Date" and not date:
                date = parse_date(cell)
            elif hdr in INITIALS and cell not in skip:
                if not date or not in_date_range(date, start_date, end_date):
                    continue
                if "1" in cell:
                    telnr: str | int = 1
                elif "2" in cell:
                    telnr = 2
                else:
                    telnr = "X"
                typ = code.get(cell, "")
                out.append(sql_insert(date, telnr, INITIALS[hdr], typ))

    return out


def convert_na(lines: list[str], verbose: bool, start_date=None, end_date=None) -> list[str]:
    skip = {"", "X", "x", "L", "SD", "sd", "HQ", "hq", "CPR", "cpr", "MT", "mt", "PD"}
    for i in range(1, 100):
        skip.add(f"L{i}")

    header: list[str] = []
    out: list[str] = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if verbose:
            print(f"NA line: {line}", file=sys.stderr, flush=True)
        line_parts = parse_csv_line(line)

        if "DOW,Date" in line:
            header = [h for h in line_parts if h]
            continue

        date = ""
        for key, hdr in enumerate(header):
            if key >= len(line_parts):
                continue
            cell = line_parts[key]
            if hdr == "Date" and not date:
                date = parse_date(cell)
            elif hdr in INITIALS and cell not in skip:
                if not date or not in_date_range(date, start_date, end_date):
                    continue
                typ = cell.lower()
                out.append(sql_insert(date, 0, INITIALS[hdr], typ))

    return out


def convert_sa(lines: list[str], verbose: bool, start_date=None, end_date=None) -> list[str]:
    skip = {"", "-"}
    out: list[str] = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if verbose:
            print(f"SA line: {line}", file=sys.stderr, flush=True)
        if "Date" in line:
            continue

        line_parts = parse_csv_line(line)
        if not line_parts:
            continue

        date = parse_date(line_parts[0])
        if not in_date_range(date, start_date, end_date):
            continue

        # Split each telnr column independently so splits do not cross columns.
        for telnr in (1, 2):
            if telnr >= len(line_parts):
                continue

            raw_cell = line_parts[telnr]
            if "/" in raw_cell:
                split_result = parse_split_cell.parse_split_row([raw_cell])
                expanded_rows = split_result.get("rows")
                if expanded_rows is None:
                    expanded_rows = [[raw_cell]]
            else:
                expanded_rows = [[raw_cell]]

            for expanded_row in expanded_rows:
                if not expanded_row:
                    continue
                cell = expanded_row[0].strip()
                if cell in skip:
                    continue
                oncall = "oc" if "oc" in cell else ""
                initials = cell.replace("oc", "")
                alias = INITIALS_SA.get(initials, "")
                typ = f"sa{oncall}"
                out.append(sql_insert(date, telnr, alias, typ))

    return out


# disregard EEOC conversion - skip

# def convert_eeoc(lines: list[str], verbose: bool) -> list[str]:
#     out: list[str] = []
#     date_key: int | None = None
#     alias_key: int | None = None

#     for line in lines:
#         line = line.strip()
#         if not line:
#             continue
#         if verbose:
#             print(f"EEOC line: {line}", file=sys.stderr, flush=True)

#         split = parse_csv_line(line)
#         if "Date" in split:
#             date_key = split.index("Date")
#             alias_key = split.index("Alias") if "Alias" in split else None
#             continue

#         if date_key is None or alias_key is None:
#             continue
#         if date_key >= len(split) or alias_key >= len(split):
#             continue

#         date_value = split[date_key]
#         first_space = date_value.find(" ")
#         if first_space != -1:
#             date_value = date_value[first_space + 1 :].strip()
#         date = parse_date(date_value)

#         alias = split[alias_key].strip()
#         if alias:
#             out.append(sql_insert(date, 0, alias, "eeoc"))

#     return out


In [56]:
def main(start_date: str | None = None, end_date: str | None = None) -> None:

    # --- option 1: arguments for API call (with config file) ---

    # --- option 2: arguments from command line (standalone call) ---
    # args = parse_args()
    # if not args.input.exists():
    #     raise FileNotFoundError(f"Input file does not exist: {args.input}")


    # --- option 3: arguments hardcoded here (run or test) ---

    # ===== begin user configuration =====

    # 1. select input mode
    verbose = True  # Set verbose to True for debugging output

    # 2. select schedule type (uncomment only one of the following lines)
    # type = 'sa'     # staff astronomer
    # type = 'oa'     # observing assistant
    # type = 'na'     # night attendant
    type = 'eeoc'   # electrical engineering on call (default, do not use)

    if not type:
        print('Please select a schedule type: "sa", "oa", or "na"')
        return

    # 3. specify date range filter (inclusive) as YYYY-MM-DD
    # all rows outside this range are ignored
    # uncomment only one set of start_date and end_date for testing or production use

    start_date = None          # test
    end_date = None

    # start_date = "2026-08-01"  # SA (343)   split-cells: 8/8, 9/26, 12/16, 01/04/27
    # end_date = "2027-01-31"

    # start_date = "2026-07-24"  # OA (259)
    # end_date = "2027-01-31"

    # start_date = "2026-08-01"  # NA (72)
    # end_date = "2026-10-31"

    # start_date = "2026-08-01"  # custom range for testing
    # end_date = "2027-01-31"

    # ===== end user configuration =====

    start_date_obj = parse_date_to_obj(start_date) if start_date else None
    end_date_obj = parse_date_to_obj(end_date) if end_date else None
    if start_date_obj and end_date_obj and start_date_obj > end_date_obj:
        raise ValueError("start_date must be earlier than or equal to end_date")

    # Optionally specify full paths here. Leave as None to use defaults.
    input_file = None  # Example: Path("../input_files/2026B_OA_Staff_Schedule.csv")
    output_file = None  # Example: Path("../output_files/custom_output.sql")

    if input_file is None:
        match type:
            case 'sa':
                input_file = Path("../input_files/2026B_SA_Staff_Schedule.csv")
            case 'oa':
                input_file = Path("../input_files/2026B_OA_Staff_Schedule.csv")
            case 'na':
                input_file = Path("../input_files/2026B_NA_Staff_Schedule.csv")
            case 'eeoc':
                print('EEOC is not supported. Please use the --type option with "sa", "oa", or "na".')
                return
            case _:
                print(f'Unsupported type: {type}')
                return

    if not input_file.exists():
        raise FileNotFoundError(f"Input file does not exist: {input_file}")
    
    input_file = Path(input_file)
    if input_file.suffix.lower() != ".csv":
        raise ValueError(f"Input file must be a CSV: {input_file}")

    if output_file is None:
        # Generate output filename from input filename by replacing .csv with .sql
        output_file = Path("../output_files") / input_file.with_suffix(".sql").name
    else:
        output_file = Path(output_file)



    print(f" Input file: {input_file}")
    print(f"Output file: {output_file}")
    print(f"Date range: {start_date_obj} to {end_date_obj}")

    lines = read_lines(input_file)
    for line in lines:
        if line.strip():
            print(f"First non-empty line: {line}")
            break

    converters = {
        "oa": convert_oa,
        "na": convert_na,
        "sa": convert_sa,
        # "eeoc": convert_eeoc,
    }

    import inspect

    converter = converters[type]
    param_count = len(inspect.signature(converter).parameters)
    if param_count >= 4:
        sql_lines = converter(lines, verbose, start_date_obj, end_date_obj)
    else:
        sql_lines = converter(lines, verbose)

    generate_output(sql_lines, output_file)

    print("\nConversion completed!\n")

In [58]:
# executes the main function
main()

EEOC is not supported. Please use the --type option with "sa", "oa", or "na".


<!-- if __name__ == "__main__":
    main() -->